# Linear-feature LASSO cold-start validation

This notebook evaluates a fixed linear descriptor set with a LASSO model under three out-of-sample regimes:

1. **LOCO:** leave one catalyst out.
2. **LOSO:** leave one substrate out.
3. **Double cold start:** leave both the catalyst and substrate of a target combination out of training.

Only `training_set_base.csv` is used. The external test set is intentionally not loaded. Scaling and LASSO regularization selection are repeated using only the training portion of every outer split.


## Validation design

For each outer split, the notebook performs an inner cross-validation on the outer-training data to select the regularization strength. The largest alpha within one standard error of the minimum inner-CV MSE is used. The selected model is then refitted on the complete outer-training set and evaluated only on the held-out observations.

The resulting predictions are therefore outer out-of-fold predictions. Metrics in the final summary are calculated from the aggregated held-out predictions, not from training predictions.

The linear-feature curation is treated as fixed. No SISSO, Boruta, or response-guided feature selection is performed in this baseline notebook.


In [ ]:
from pathlib import Path
import json
import platform
import re
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso, lasso_path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_STATE = 42
INNER_N_SPLITS = 5
ALPHA_RATIOS = np.logspace(0, -4, 80)  # alpha_max down to 1e-4 * alpha_max.
USE_ONE_STANDARD_ERROR_RULE = True
MAX_DOUBLE_COLD_START_FOLDS = None  # Set to an integer for a quick test run.
SAVE_RESULTS = True

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
def locate_training_file():
    candidates = [
        Path("training_set_base.csv"),
        Path("baseline_validation/training_set_base.csv"),
        Path("catalyst-substrate-modeling/baseline_validation/training_set_base.csv"),
        Path("linear_features/training_set_base.csv"),
        Path("catalyst-substrate-modeling/linear_features/training_set_base.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find training_set_base.csv. Run the notebook from the repository root, "
        "catalyst-substrate-modeling, or baseline_validation directory."
    )


DATA_FILE = locate_training_file()
OUTPUT_DIR = DATA_FILE.parent / "results" / "cold_start_linear_lasso"
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training data: {DATA_FILE}")
print(f"Results directory: {OUTPUT_DIR}")


## Load and validate the fixed linear feature table

In [ ]:
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")

ID_COLUMN = "cat_substrate"
TARGET_COLUMN = "ddG"
required_columns = {ID_COLUMN, TARGET_COLUMN}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

identifier_parts = df[ID_COLUMN].astype(str).str.rsplit("_", n=1, expand=True)
if identifier_parts.shape[1] != 2 or identifier_parts.isna().any().any():
    raise ValueError("Each cat_substrate identifier must end with '_<substrate>'.")

df = df.copy()
df["catalyst"] = identifier_parts[0].to_numpy()
df["substrate"] = identifier_parts[1].to_numpy()

feature_columns = [column for column in df.columns if re.fullmatch(r"x\d+", str(column))]
feature_columns = sorted(feature_columns, key=lambda name: int(name[1:]))
if not feature_columns:
    raise ValueError("No linear descriptor columns matching x<number> were found.")

X = df[feature_columns].astype(float).to_numpy()
y = df[TARGET_COLUMN].astype(float).to_numpy()
meta = df[[ID_COLUMN, "catalyst", "substrate"]].reset_index(drop=True)

if not np.isfinite(X).all():
    raise ValueError("The feature matrix contains missing or non-finite values.")
if not np.isfinite(y).all():
    raise ValueError("The response vector contains missing or non-finite values.")
if meta[ID_COLUMN].duplicated().any():
    duplicates = meta.loc[meta[ID_COLUMN].duplicated(keep=False), ID_COLUMN].tolist()
    raise ValueError(f"Duplicate catalyst-substrate identifiers found: {duplicates[:10]}")

dataset_summary = pd.Series(
    {
        "observations": len(df),
        "catalysts": meta["catalyst"].nunique(),
        "substrates": meta["substrate"].nunique(),
        "linear_features": len(feature_columns),
        "target_min": y.min(),
        "target_max": y.max(),
    },
    name="value",
)
display(dataset_summary.to_frame())
display(df[[ID_COLUMN, TARGET_COLUMN, "catalyst", "substrate"] + feature_columns[:5]].head())


## Define the outer cold-start splits

For double cold start, each observed catalyst-substrate combination is predicted after removing every training row that contains either its catalyst or its substrate. Rows containing only one of the two held-out components are not used in that outer fold.


In [ ]:
def make_loco_splits(meta_df):
    splits = []
    for catalyst in sorted(meta_df["catalyst"].unique()):
        test_mask = meta_df["catalyst"].eq(catalyst).to_numpy()
        splits.append(
            {
                "regime": "LOCO",
                "fold_id": f"catalyst={catalyst}",
                "train_idx": np.flatnonzero(~test_mask),
                "test_idx": np.flatnonzero(test_mask),
                "held_out_catalyst": catalyst,
                "held_out_substrate": None,
            }
        )
    return splits


def make_loso_splits(meta_df):
    splits = []
    for substrate in sorted(meta_df["substrate"].unique()):
        test_mask = meta_df["substrate"].eq(substrate).to_numpy()
        splits.append(
            {
                "regime": "LOSO",
                "fold_id": f"substrate={substrate}",
                "train_idx": np.flatnonzero(~test_mask),
                "test_idx": np.flatnonzero(test_mask),
                "held_out_catalyst": None,
                "held_out_substrate": substrate,
            }
        )
    return splits


def make_double_cold_start_splits(meta_df, max_folds=None):
    splits = []
    unique_pairs = meta_df[["catalyst", "substrate"]].drop_duplicates()
    if max_folds is not None:
        unique_pairs = unique_pairs.head(max_folds)

    for catalyst, substrate in unique_pairs.itertuples(index=False, name=None):
        test_mask = (
            meta_df["catalyst"].eq(catalyst)
            & meta_df["substrate"].eq(substrate)
        ).to_numpy()
        train_mask = (
            ~meta_df["catalyst"].eq(catalyst)
            & ~meta_df["substrate"].eq(substrate)
        ).to_numpy()
        train_idx = np.flatnonzero(train_mask)
        test_idx = np.flatnonzero(test_mask)
        if len(train_idx) == 0 or len(test_idx) == 0:
            continue
        splits.append(
            {
                "regime": "double_cold_start",
                "fold_id": f"catalyst={catalyst}|substrate={substrate}",
                "train_idx": train_idx,
                "test_idx": test_idx,
                "held_out_catalyst": catalyst,
                "held_out_substrate": substrate,
            }
        )
    return splits


outer_splits = {
    "LOCO": make_loco_splits(meta),
    "LOSO": make_loso_splits(meta),
    "double_cold_start": make_double_cold_start_splits(
        meta, max_folds=MAX_DOUBLE_COLD_START_FOLDS
    ),
}

pd.DataFrame(
    {
        "regime": list(outer_splits),
        "outer_folds": [len(splits) for splits in outer_splits.values()],
        "total_test_predictions": [
            sum(len(split["test_idx"]) for split in splits)
            for splits in outer_splits.values()
        ],
    }
)


In [ ]:
def audit_outer_splits(split_dict, meta_df):
    for regime, splits in split_dict.items():
        for split in splits:
            train_meta = meta_df.iloc[split["train_idx"]]
            test_meta = meta_df.iloc[split["test_idx"]]
            if regime == "LOCO":
                assert set(train_meta["catalyst"]).isdisjoint(test_meta["catalyst"])
            elif regime == "LOSO":
                assert set(train_meta["substrate"]).isdisjoint(test_meta["substrate"])
            elif regime == "double_cold_start":
                assert set(train_meta["catalyst"]).isdisjoint(test_meta["catalyst"])
                assert set(train_meta["substrate"]).isdisjoint(test_meta["substrate"])
            else:
                raise ValueError(f"Unknown regime: {regime}")

    print("Outer-split audit passed: no held-out identities occur in their training folds.")


audit_outer_splits(outer_splits, meta)


## Inner validation and the 1-SE alpha rule

LOCO uses catalyst-grouped inner folds and LOSO uses substrate-grouped inner folds. Double-cold-start models use inner folds in which both catalyst blocks and substrate blocks are omitted from the corresponding inner-training partition.


In [ ]:
def grouped_inner_splits(groups, n_splits=5):
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    effective_splits = min(n_splits, len(unique_groups))
    if effective_splits < 2:
        raise ValueError("At least two groups are required for grouped inner validation.")
    splitter = GroupKFold(n_splits=effective_splits)
    dummy_X = np.zeros((len(groups), 1))
    return list(splitter.split(dummy_X, groups=groups))


def double_cold_start_inner_splits(meta_train, n_splits=5, random_state=42):
    catalysts = np.array(sorted(meta_train["catalyst"].unique()), dtype=object)
    substrates = np.array(sorted(meta_train["substrate"].unique()), dtype=object)
    effective_splits = min(n_splits, len(catalysts), len(substrates))
    if effective_splits < 2:
        raise ValueError("At least two catalysts and two substrates are required.")

    rng = np.random.default_rng(random_state)
    catalysts = rng.permutation(catalysts)
    substrates = rng.permutation(substrates)
    catalyst_blocks = np.array_split(catalysts, effective_splits)
    substrate_blocks = np.array_split(substrates, effective_splits)

    splits = []
    # Try different catalyst/substrate block pairings until enough non-empty folds are found.
    for offset in range(effective_splits):
        for fold_number in range(effective_splits):
            held_catalysts = set(catalyst_blocks[fold_number])
            held_substrates = set(
                substrate_blocks[(fold_number + offset) % effective_splits]
            )
            train_mask = (
                ~meta_train["catalyst"].isin(held_catalysts)
                & ~meta_train["substrate"].isin(held_substrates)
            ).to_numpy()
            validation_mask = (
                meta_train["catalyst"].isin(held_catalysts)
                & meta_train["substrate"].isin(held_substrates)
            ).to_numpy()
            train_idx = np.flatnonzero(train_mask)
            validation_idx = np.flatnonzero(validation_mask)
            if len(train_idx) >= 2 and len(validation_idx) >= 1:
                splits.append((train_idx, validation_idx))
            if len(splits) >= effective_splits:
                return splits

    if len(splits) < 2:
        # Leakage-safe fallback if the observed matrix is too sparse for block intersections.
        fallback = KFold(
            n_splits=min(effective_splits, len(meta_train)),
            shuffle=True,
            random_state=random_state,
        )
        return list(fallback.split(np.zeros((len(meta_train), 1))))
    return splits


def make_inner_splits(regime, meta_train):
    if regime == "LOCO":
        return grouped_inner_splits(meta_train["catalyst"], INNER_N_SPLITS)
    if regime == "LOSO":
        return grouped_inner_splits(meta_train["substrate"], INNER_N_SPLITS)
    if regime == "double_cold_start":
        return double_cold_start_inner_splits(
            meta_train, INNER_N_SPLITS, RANDOM_STATE
        )
    raise ValueError(f"Unknown validation regime: {regime}")


In [ ]:
def calculate_alpha_max(X_scaled, y_values):
    # alpha_max is the smallest LASSO penalty that yields an intercept-only model.
    y_centered = y_values - y_values.mean()
    alpha_max = np.max(np.abs(X_scaled.T @ y_centered)) / len(y_values)
    if not np.isfinite(alpha_max) or alpha_max <= 0:
        raise ValueError("Could not construct a positive LASSO alpha grid.")
    return float(alpha_max)


def select_lasso_alpha_ratio(X_train, y_train, inner_splits, alpha_ratios):
    fold_mse = np.full((len(alpha_ratios), len(inner_splits)), np.nan, dtype=float)

    for split_number, (inner_train_idx, inner_validation_idx) in enumerate(inner_splits):
        scaler = StandardScaler()
        X_inner_train = scaler.fit_transform(X_train[inner_train_idx])
        X_inner_validation = scaler.transform(X_train[inner_validation_idx])

        y_inner_train = y_train[inner_train_idx]
        y_inner_mean = y_inner_train.mean()
        inner_alpha_max = calculate_alpha_max(X_inner_train, y_inner_train)
        inner_alphas = inner_alpha_max * alpha_ratios
        path_alphas, path_coefficients, _ = lasso_path(
            X_inner_train,
            y_inner_train - y_inner_mean,
            alphas=inner_alphas,
            max_iter=100_000,
            tol=1e-6,
        )
        if not np.allclose(path_alphas, inner_alphas):
            raise RuntimeError("lasso_path returned an unexpected alpha ordering.")
        prediction_matrix = X_inner_validation @ path_coefficients + y_inner_mean
        validation_residuals = (
            y_train[inner_validation_idx, np.newaxis] - prediction_matrix
        )
        fold_mse[:, split_number] = np.mean(validation_residuals**2, axis=0)

    mean_mse = np.nanmean(fold_mse, axis=1)
    if fold_mse.shape[1] > 1:
        standard_error = np.nanstd(fold_mse, axis=1, ddof=1) / np.sqrt(
            np.sum(np.isfinite(fold_mse), axis=1)
        )
    else:
        standard_error = np.zeros_like(mean_mse)

    minimum_index = int(np.nanargmin(mean_mse))
    ratio_min = float(alpha_ratios[minimum_index])
    threshold = mean_mse[minimum_index] + standard_error[minimum_index]
    eligible = np.flatnonzero(mean_mse <= threshold)
    ratio_1se = float(np.max(alpha_ratios[eligible]))
    selected_ratio = ratio_1se if USE_ONE_STANDARD_ERROR_RULE else ratio_min

    curve = pd.DataFrame(
        {
            "alpha_ratio": alpha_ratios,
            "mean_validation_mse": mean_mse,
            "standard_error": standard_error,
        }
    )
    return {
        "ratio_min": ratio_min,
        "ratio_1se": ratio_1se,
        "selected_ratio": selected_ratio,
        "curve": curve,
    }


## Fit and evaluate every outer split

In [ ]:
def safe_r2(y_true, y_pred):
    if len(y_true) < 2 or np.allclose(y_true, y_true[0]):
        return np.nan
    return r2_score(y_true, y_pred)


def evaluate_outer_split(split, X_all, y_all, meta_all):
    train_idx = split["train_idx"]
    test_idx = split["test_idx"]
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]
    meta_train = meta_all.iloc[train_idx].reset_index(drop=True)

    inner_splits = make_inner_splits(split["regime"], meta_train)
    alpha_result = select_lasso_alpha_ratio(
        X_train, y_train, inner_splits, ALPHA_RATIOS
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    outer_alpha_max = calculate_alpha_max(X_train_scaled, y_train)
    alpha_min = outer_alpha_max * alpha_result["ratio_min"]
    alpha_1se = outer_alpha_max * alpha_result["ratio_1se"]
    selected_alpha = outer_alpha_max * alpha_result["selected_ratio"]
    model = Lasso(
        alpha=selected_alpha,
        max_iter=100_000,
        tol=1e-6,
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    prediction_table = meta_all.iloc[test_idx].copy()
    prediction_table["row_index"] = test_idx
    prediction_table["regime"] = split["regime"]
    prediction_table["fold_id"] = split["fold_id"]
    prediction_table["y_true"] = y_test
    prediction_table["y_pred"] = y_pred
    prediction_table["residual"] = y_test - y_pred
    prediction_table["absolute_error"] = np.abs(y_test - y_pred)
    prediction_table["selected_alpha"] = selected_alpha
    prediction_table["selected_alpha_fraction_of_max"] = alpha_result[
        "selected_ratio"
    ]

    fold_metrics = {
        "regime": split["regime"],
        "fold_id": split["fold_id"],
        "held_out_catalyst": split["held_out_catalyst"],
        "held_out_substrate": split["held_out_substrate"],
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_inner_splits": len(inner_splits),
        "alpha_grid_max": outer_alpha_max,
        "alpha_min": alpha_min,
        "alpha_1se": alpha_1se,
        "selected_alpha": selected_alpha,
        "selected_alpha_fraction_of_max": alpha_result["selected_ratio"],
        "n_nonzero": int(np.count_nonzero(model.coef_)),
        "r2": safe_r2(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": mean_squared_error(y_test, y_pred) ** 0.5,
    }
    return prediction_table, fold_metrics


In [ ]:
all_predictions = []
all_fold_metrics = []
start_time = time.time()

for regime, splits in outer_splits.items():
    regime_start = time.time()
    print(f"Running {regime}: {len(splits)} outer folds")
    for fold_number, split in enumerate(splits, start=1):
        predictions, fold_metrics = evaluate_outer_split(split, X, y, meta)
        all_predictions.append(predictions)
        all_fold_metrics.append(fold_metrics)
        if fold_number == 1 or fold_number % 20 == 0 or fold_number == len(splits):
            print(f"  completed {fold_number}/{len(splits)} folds")
    print(f"  elapsed: {time.time() - regime_start:.1f} s")

predictions_df = pd.concat(all_predictions, ignore_index=True)
fold_metrics_df = pd.DataFrame(all_fold_metrics)
print(f"Total elapsed: {time.time() - start_time:.1f} s")


## Aggregate the outer out-of-fold metrics

For LOCO and LOSO, every observation should receive one held-out prediction. In the complete double-cold-start run, every observed catalyst-substrate combination should also receive one prediction. R² is calculated only after aggregating all held-out predictions within a regime.


In [ ]:
def aggregate_metrics(prediction_table, fold_table):
    rows = []
    for regime, regime_predictions in prediction_table.groupby("regime", sort=False):
        regime_folds = fold_table.loc[fold_table["regime"].eq(regime)]
        y_true = regime_predictions["y_true"].to_numpy()
        y_pred = regime_predictions["y_pred"].to_numpy()
        rows.append(
            {
                "regime": regime,
                "n_outer_folds": len(regime_folds),
                "n_predictions": len(regime_predictions),
                "n_unique_rows_predicted": regime_predictions["row_index"].nunique(),
                "r2_oof": safe_r2(y_true, y_pred),
                "mae_oof": mean_absolute_error(y_true, y_pred),
                "rmse_oof": mean_squared_error(y_true, y_pred) ** 0.5,
                "median_absolute_error": regime_predictions["absolute_error"].median(),
                "median_selected_alpha": regime_folds["selected_alpha"].median(),
                "median_alpha_fraction_of_max": regime_folds[
                    "selected_alpha_fraction_of_max"
                ].median(),
                "median_nonzero_features": regime_folds["n_nonzero"].median(),
            }
        )
    return pd.DataFrame(rows)


summary_metrics_df = aggregate_metrics(predictions_df, fold_metrics_df)

coverage = predictions_df.groupby("regime")["row_index"].agg(["count", "nunique"])
if MAX_DOUBLE_COLD_START_FOLDS is None:
    assert (coverage["count"] == len(df)).all(), coverage
    assert (coverage["nunique"] == len(df)).all(), coverage

display(
    summary_metrics_df.style.format(
        {
            "r2_oof": "{:.3f}",
            "mae_oof": "{:.3f}",
            "rmse_oof": "{:.3f}",
            "median_absolute_error": "{:.3f}",
            "median_selected_alpha": "{:.5f}",
            "median_alpha_fraction_of_max": "{:.4f}",
            "median_nonzero_features": "{:.1f}",
        }
    )
)


In [ ]:
regime_order = ["LOCO", "LOSO", "double_cold_start"]
figure, axes = plt.subplots(1, 3, figsize=(14, 4.3), constrained_layout=True)
all_values = np.concatenate([predictions_df["y_true"], predictions_df["y_pred"]])
plot_min, plot_max = all_values.min(), all_values.max()
padding = 0.05 * (plot_max - plot_min)
limits = (plot_min - padding, plot_max + padding)

for axis, regime in zip(axes, regime_order):
    subset = predictions_df.loc[predictions_df["regime"].eq(regime)]
    metrics = summary_metrics_df.loc[summary_metrics_df["regime"].eq(regime)].iloc[0]
    axis.scatter(
        subset["y_true"], subset["y_pred"], s=32, alpha=0.75, edgecolor="none"
    )
    axis.plot(limits, limits, linestyle="--", color="black", linewidth=1)
    axis.set_xlim(limits)
    axis.set_ylim(limits)
    axis.set_aspect("equal", adjustable="box")
    plot_title = {
        "LOCO": "LOCO",
        "LOSO": "LOSO",
        "double_cold_start": "Double cold start",
    }[regime]
    axis.set_title(plot_title)
    axis.set_xlabel(r"Measured $\Delta\Delta G^{\ddagger}$ (kcal mol$^{-1}$)")
    axis.set_ylabel(r"Predicted $\Delta\Delta G^{\ddagger}$ (kcal mol$^{-1}$)")
    axis.text(
        0.04,
        0.96,
        f"$R^2$ = {metrics['r2_oof']:.3f}\nMAE = {metrics['mae_oof']:.3f}\nRMSE = {metrics['rmse_oof']:.3f}",
        transform=axis.transAxes,
        va="top",
        ha="left",
        fontsize=9,
    )

if SAVE_RESULTS:
    figure.savefig(OUTPUT_DIR / "cold_start_parity.png", dpi=300, bbox_inches="tight")
    figure.savefig(OUTPUT_DIR / "cold_start_parity.pdf", bbox_inches="tight")
plt.show()


In [ ]:
largest_errors_df = (
    predictions_df.sort_values(["regime", "absolute_error"], ascending=[True, False])
    .groupby("regime", sort=False)
    .head(10)
    [[
        "regime",
        ID_COLUMN,
        "y_true",
        "y_pred",
        "absolute_error",
        "selected_alpha",
    ]]
)
display(largest_errors_df)


## Save the validation report

The prediction-level table is the most important output: it allows every metric and plot to be reconstructed without refitting the models.


In [ ]:
run_metadata = {
    "data_file": str(DATA_FILE),
    "n_observations": len(df),
    "n_catalysts": int(meta["catalyst"].nunique()),
    "n_substrates": int(meta["substrate"].nunique()),
    "feature_columns": feature_columns,
    "target_column": TARGET_COLUMN,
    "random_state": RANDOM_STATE,
    "inner_n_splits": INNER_N_SPLITS,
    "alpha_ratios": ALPHA_RATIOS.tolist(),
    "alpha_grid_definition": (
        "alpha/alpha_max ratios from 1 to 1e-4; alpha_max is recalculated "
        "inside every inner-training fold and on every outer-training fold"
    ),
    "alpha_rule": "1-SE" if USE_ONE_STANDARD_ERROR_RULE else "minimum MSE",
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
}

if SAVE_RESULTS:
    predictions_df.to_csv(OUTPUT_DIR / "cold_start_predictions.csv", index=False)
    fold_metrics_df.to_csv(OUTPUT_DIR / "cold_start_fold_metrics.csv", index=False)
    summary_metrics_df.to_csv(OUTPUT_DIR / "cold_start_summary_metrics.csv", index=False)
    largest_errors_df.to_csv(OUTPUT_DIR / "cold_start_largest_errors.csv", index=False)
    with (OUTPUT_DIR / "run_metadata.json").open("w", encoding="utf-8") as handle:
        json.dump(run_metadata, handle, indent=2)
    print(f"Saved report to {OUTPUT_DIR}")

summary_metrics_df


## Interpretation notes

- LOCO measures transfer to a catalyst not present in training.
- LOSO measures transfer to a substrate not present in training.
- Double cold start measures transfer when neither component is present in training.
- The double-cold-start result will normally be the most difficult and should not be compared with random row-wise cross-validation.
- These results describe the fixed linear-feature LASSO baseline. Later models should reuse the same outer split definitions and prediction-level reporting structure.
